<a href="https://colab.research.google.com/github/2873991-Manthena/Dissertation/blob/main/TextCNN_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 ###############################################################
# SECTION 1 - INSTALL LIBRARIES
###############################################################

!pip install -q transformers datasets accelerate

In [ ]:
###############################################################
# SECTION 2 - IMPORT LIBRARIES
###############################################################

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
###############################################################
# SECTION 3 - MOUNT GOOGLE DRIVE
###############################################################

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
###############################################################
# SECTION 4 - DATASET PATHS
###############################################################

MBTI1_PATH = "/content/drive/MyDrive/Datasets/mbti1_preprocessed.csv"

MBTI500_PATH = "/content/drive/MyDrive/Datasets/mbti500_preprocessed.csv"

COMBINED_PATH = "/content/drive/MyDrive/Datasets/combined_mbti_preprocessed.csv"

In [ ]:
###############################################################
# SECTION 5 - LOAD PREPROCESSED DATASETS
###############################################################

df_mbti1 = pd.read_csv(MBTI1_PATH)
df_mbti500 = pd.read_csv(MBTI500_PATH)
df_combined = pd.read_csv(COMBINED_PATH)

print("===== MBTI1 =====")
print(df_mbti1.shape)
print(df_mbti1.columns.tolist())

print("\n===== MBTI500 =====")
print(df_mbti500.shape)
print(df_mbti500.columns.tolist())

print("\n===== COMBINED =====")
print(df_combined.shape)
print(df_combined.columns.tolist())

===== MBTI1 =====
(8675, 2)
['type', 'clean_text']

===== MBTI500 =====
(106067, 2)
['type', 'clean_text']

===== COMBINED =====
(114741, 3)
['type', 'clean_text', 'label']


In [ ]:
###############################################################
# SECTION 6 - DATA CHECK
###############################################################

display(df_mbti1.head())
display(df_mbti500.head())
display(df_combined.head())

,type,clean_text
0,INFJ,intj moment sportscenter top ten play prank li...
1,ENTP,I find lack I post alarm sex boring position o...
2,INTP,good one course I say I know my blessing my cu...
3,INTJ,dear intp I enjoy our conversation day esoteri...
4,ENTJ,you fire another silly misconception approach ...


,type,clean_text
0,INTJ,know intj tool use interaction people excuse a...
1,INTJ,rap music ehh opp yeah know valid well know fa...
2,INTJ,preferably p hd low except wew lad video p min...
3,INTJ,drink like wish could drink red wine give head...
4,INTJ,space program ah bad deal meing freelance max ...


,type,clean_text,label
0,ENTP,possibly ironic word mug also mug existence li...,3.0
1,INFP,aww man you poor poor guy I say I infp friend ...,NaN
2,ENTP,offer date marion heroic maid marion prioritiz...,3.0
3,INTP,resus may vary simple way expose large branch ...,11.0
4,INTJ,anything interest context whether contrast soc...,10.0


In [ ]:
###############################################################
# SECTION 7 - MISSING VALUES
###############################################################

print("MBTI1 missing:")
print(df_mbti1.isnull().sum())

print("\nMBTI500 missing:")
print(df_mbti500.isnull().sum())

print("\nCombined missing:")
print(df_combined.isnull().sum())

MBTI1 missing:
type          0
clean_text    1
dtype: int64

MBTI500 missing:
type          0
clean_text    0
dtype: int64

Combined missing:
type             0
clean_text       0
label         8674
dtype: int64


In [ ]:
###############################################################
# SECTION 9 - PREPARE ALL DATASETS + LABEL ENCODING
###############################################################

# Keep only the columns we need
df_mbti1 = df_mbti1[["type", "clean_text"]].dropna().copy()

df_mbti500 = df_mbti500[["type", "clean_text"]].dropna().copy()

df_combined = df_combined[["type", "clean_text"]].dropna().copy()


# Make sure text is string
df_mbti1["clean_text"] = df_mbti1["clean_text"].astype(str)
df_mbti500["clean_text"] = df_mbti500["clean_text"].astype(str)
df_combined["clean_text"] = df_combined["clean_text"].astype(str)


# Create ONE common label encoder for all datasets
label_encoder = LabelEncoder()

label_encoder.fit(
    pd.concat([
        df_mbti1["type"],
        df_mbti500["type"],
        df_combined["type"]
    ])
)


# Encode labels
df_mbti1["label"] = label_encoder.transform(
    df_mbti1["type"]
)

df_mbti500["label"] = label_encoder.transform(
    df_mbti500["type"]
)

df_combined["label"] = label_encoder.transform(
    df_combined["type"]
)


print("===== DATASET SHAPES =====")
print("MBTI1     :", df_mbti1.shape)
print("MBTI500   :", df_mbti500.shape)
print("Combined  :", df_combined.shape)

print("\n===== LABELS =====")
print(label_encoder.classes_)

print("\nNumber of classes:", len(label_encoder.classes_))

print("\n===== MISSING VALUES AFTER PREPARATION =====")
print("MBTI1:")
print(df_mbti1.isnull().sum())

print("\nMBTI500:")
print(df_mbti500.isnull().sum())

print("\nCombined:")
print(df_combined.isnull().sum())

===== DATASET SHAPES =====
MBTI1     : (8674, 3)
MBTI500   : (106067, 3)
Combined  : (114741, 3)

===== LABELS =====
['ENFJ' 'ENFP' 'ENTJ' 'ENTP' 'ESFJ' 'ESFP' 'ESTJ' 'ESTP' 'INFJ' 'INFP'
 'INTJ' 'INTP' 'ISFJ' 'ISFP' 'ISTJ' 'ISTP']

Number of classes: 16

===== MISSING VALUES AFTER PREPARATION =====
MBTI1:
type          0
clean_text    0
label         0
dtype: int64

MBTI500:
type          0
clean_text    0
label         0
dtype: int64

Combined:
type          0
clean_text    0
label         0
dtype: int64


In [ ]:
###############################################################
# SECTION 11 - TEXTCNN CONFIGURATION
###############################################################

from collections import Counter

MAX_VOCAB_SIZE = 30000
MAX_LENGTH = 256
EMBEDDING_DIM = 128
BATCH_SIZE = 64
EPOCHS = 5

print("TextCNN configuration ready!")

TextCNN configuration ready!


In [ ]:
###############################################################
# SECTION 12 - BUILD VOCABULARY
###############################################################

def tokenize_text(text):
    return text.lower().split()


counter = Counter()

# Build vocabulary from ALL preprocessed training text
for dataset in [df_mbti1, df_mbti500, df_combined]:
    for text in dataset["clean_text"]:
        counter.update(tokenize_text(text))


most_common_words = counter.most_common(
    MAX_VOCAB_SIZE - 2
)

word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for idx, (word, _) in enumerate(
    most_common_words,
    start=2
):
    word2idx[word] = idx


VOCAB_SIZE = len(word2idx)

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 30000


In [ ]:
###############################################################
# SECTION 13 - TEXT ENCODER
###############################################################

def encode_text(text):

    tokens = tokenize_text(text)

    ids = [
        word2idx.get(
            token,
            word2idx["<UNK>"]
        )
        for token in tokens
    ]

    ids = ids[:MAX_LENGTH]

    if len(ids) < MAX_LENGTH:
        ids += [0] * (
            MAX_LENGTH - len(ids)
        )

    return ids

In [ ]:
###############################################################
# SECTION 14 - PYTORCH DATASET
###############################################################

class MBTIDataset(Dataset):

    def __init__(self, texts, labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        input_ids = torch.tensor(
            encode_text(self.texts[idx]),
            dtype=torch.long
        )

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return {
            "input_ids": input_ids,
            "labels": label
        }

In [ ]:
###############################################################
# SECTION 15 - TEXTCNN MODEL
###############################################################

class TextCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_classes
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.convs = nn.ModuleList([

            nn.Conv1d(
                embedding_dim,
                128,
                kernel_size=3
            ),

            nn.Conv1d(
                embedding_dim,
                128,
                kernel_size=4
            ),

            nn.Conv1d(
                embedding_dim,
                128,
                kernel_size=5
            )
        ])

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(
            128 * 3,
            num_classes
        )

    def forward(self, input_ids):

        x = self.embedding(input_ids)

        # [batch, sequence, embedding]
        # -> [batch, embedding, sequence]
        x = x.permute(0, 2, 1)

        pooled_outputs = []

        for conv in self.convs:

            conv_output = torch.relu(
                conv(x)
            )

            pooled = torch.max(
                conv_output,
                dim=2
            ).values

            pooled_outputs.append(pooled)

        x = torch.cat(
            pooled_outputs,
            dim=1
        )

        x = self.dropout(x)

        return self.fc(x)

In [ ]:
###############################################################
# SECTION 16 - TEXTCNN TRAINING FUNCTION
###############################################################

def run_textcnn(
    df,
    dataset_name,
    epochs=5
):

    print("\n" + "=" * 60)
    print(f"TEXTCNN - {dataset_name}")
    print("=" * 60)

    # ---------------------------------------------------------
    # TRAIN / TEST SPLIT
    # ---------------------------------------------------------

    train_texts, test_texts, train_labels, test_labels = train_test_split(

        df["clean_text"].values,

        df["label"].values,

        test_size=0.20,

        random_state=42,

        stratify=df["label"].values
    )

    print("Training samples:", len(train_texts))
    print("Testing samples :", len(test_texts))

    # ---------------------------------------------------------
    # DATASETS
    # ---------------------------------------------------------

    train_dataset = MBTIDataset(
        train_texts,
        train_labels
    )

    test_dataset = MBTIDataset(
        test_texts,
        test_labels
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # ---------------------------------------------------------
    # MODEL
    # ---------------------------------------------------------

    model = TextCNN(
        vocab_size=VOCAB_SIZE,
        embedding_dim=EMBEDDING_DIM,
        num_classes=16
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4
    )

    # ---------------------------------------------------------
    # TRAINING
    # ---------------------------------------------------------

    start_time = time.time()

    for epoch in range(epochs):

        model.train()

        total_loss = 0

        for batch in train_loader:

            input_ids = batch["input_ids"].to(device)

            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            outputs = model(input_ids)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = (
            total_loss /
            len(train_loader)
        )

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Loss: {avg_loss:.4f}"
        )

    training_time = time.time() - start_time

    # ---------------------------------------------------------
    # EVALUATION
    # ---------------------------------------------------------

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch["input_ids"].to(device)

            labels = batch["labels"].to(device)

            outputs = model(input_ids)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    # ---------------------------------------------------------
    # METRICS
    # ---------------------------------------------------------

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    print("\n===== FINAL RESULTS =====")

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print(
        f"Training Time: "
        f"{training_time / 60:.2f} minutes"
    )

    return {
        "Dataset": dataset_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Training Time (min)": training_time / 60
    }

In [ ]:
###############################################################
# FIX - DEFINE DEVICE
###############################################################

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
###############################################################
# SECTION 17 - TEXTCNN MBTI1
###############################################################

result_mbti1 = run_textcnn(
    df_mbti1,
    "MBTI1",
    epochs=5
)


TEXTCNN - MBTI1
Training samples: 6939
Testing samples : 1735
Epoch 1/5 - Loss: 2.6785
Epoch 2/5 - Loss: 2.5009
Epoch 3/5 - Loss: 2.3814
Epoch 4/5 - Loss: 2.2744
Epoch 5/5 - Loss: 2.1492

===== FINAL RESULTS =====
Accuracy : 0.4207
Precision: 0.3984
Recall   : 0.4207
F1 Score : 0.3470
Training Time: 0.20 minutes


In [ ]:
result_mbti500 = run_textcnn(
    df_mbti500,
    "MBTI500",
    epochs=5
)


TEXTCNN - MBTI500
Training samples: 84853
Testing samples : 21214
Epoch 1/5 - Loss: 2.0221
Epoch 2/5 - Loss: 1.5918
Epoch 3/5 - Loss: 1.4692
Epoch 4/5 - Loss: 1.3984
Epoch 5/5 - Loss: 1.3462

===== FINAL RESULTS =====
Accuracy : 0.6220
Precision: 0.6132
Recall   : 0.6220
F1 Score : 0.6056
Training Time: 1.78 minutes


In [ ]:
result_combined = run_textcnn(
    df_combined,
    "Combined",
    epochs=5
)


TEXTCNN - Combined
Training samples: 91792
Testing samples : 22949
Epoch 1/5 - Loss: 2.0174
Epoch 2/5 - Loss: 1.6047
Epoch 3/5 - Loss: 1.4782
Epoch 4/5 - Loss: 1.3995
Epoch 5/5 - Loss: 1.3516

===== FINAL RESULTS =====
Accuracy : 0.6180
Precision: 0.6107
Recall   : 0.6180
F1 Score : 0.6003
Training Time: 1.87 minutes


In [ ]:
textcnn_results = pd.DataFrame([
    result_mbti1,
    result_mbti500,
    result_combined
])

display(textcnn_results)

,Dataset,Accuracy,Precision,Recall,F1 Score,Training Time (min)
0,MBTI1,0.420749,0.398450,0.420749,0.346962,0.197932
1,MBTI500,0.622042,0.613193,0.622042,0.605560,1.775029
2,Combined,0.617979,0.610707,0.617979,0.600252,1.872626


In [ ]:
textcnn_results.to_csv(
    "/content/drive/MyDrive/TextCNN_Preprocessed_Results.csv",
    index=False
)

print("✅ ALL TEXTCNN RESULTS SAVED")

✅ ALL TEXTCNN RESULTS SAVED
